In [1]:
import cultiv
import stimcirq
import cirq
import numpy as np
from collections import Counter

from IPython.display import display, HTML
width = 75
style_str = "<style>.container { width:__% !important; }</style>".replace('__', str(width))
display(HTML(style_str))

In [2]:
from cultiv._construction.full_clifford_sim.main_compiled_fxns_fault5 import full_circuit_f5
from cultiv._construction.full_clifford_sim.main_complied_fxns import full_circuit

In [3]:
import sys
sys.path.append('../MSC_foldedH/src/full_clifford_sim/')
from cirq_utilities import remove_QCs

In [4]:
final_distance = 11
fault_distance = 5
show_circuits = False

In [5]:
if fault_distance == 5:
    stim_circuit = full_circuit_f5(nm=.001, prep='hookinj', dfinal=final_distance, cultiv_only=False).without_noise()
elif fault_distance == 3:
    stim_circuit = full_circuit(nm=.001, prep='hookinj', dfinal=final_distance, cultiv_only=False).without_noise()
else:
    raise ValueError

5


In [6]:
if show_circuits:
    stim_circuit.diagram('detslice-with-ops')

In [7]:
stim_rep = remove_QCs(stim_circuit, 0.001, reverse_Y=False).without_noise()
stim_rep = stim_rep.with_inlined_feedback()
cirq_rep = stimcirq.stim_circuit_to_cirq_circuit(stim_rep)

In [8]:
if show_circuits:
    display(cirq_rep)

In [9]:
d2 = final_distance
#### process Clifford circuit into nonClifford Circ
edited_cirq_rep = cirq.Circuit()
sevendiaggate = cirq.MatrixGate(np.array([[0,1],[1j,0]]))
sodddiaggate = cirq.MatrixGate(np.array([[0,1],[-1j,0]]))

cy_moments = 0
ghzh_counter = 0

for midx, moment in enumerate(cirq_rep):
    new_moment = []
    this_moment_is_CY = False

    for op in moment:

        if op.gate == cirq.ops.SingleQubitCliffordGate(_clifford_tableau=cirq.CliffordTableau( #ONLY FOR UNITARY PREP
            1, rs=np.array([True, False]), xs=np.array([[True], [True]]), zs=np.array([[False], [True]]))):
            new_moment.append(cirq.PhasedXPowGate(phase_exponent=3/4, exponent=1/2).on(op.qubits[0]))

        elif op.gate == cirq.ControlledGate(cirq.Y): #add a CH gate layer
            if this_moment_is_CY:
                pass # print(f"{op} at moment {midx} is removed")
            else:
                ghzqub1 = op.qubits[0].x

                if cy_moments % 3 == 0: #gate layer 1: CCZs
                    ccz1 = cirq.CCZ(cirq.LineQubit(ghzqub1) , cirq.LineQubit(4), cirq.LineQubit(4*d2))
                    ccz2 = cirq.CCZ(cirq.LineQubit(ghzqub1+1) ,cirq.LineQubit(2*d2+6) ,cirq.LineQubit(6*d2+2) )
                    ccz3 = cirq.CCZ(cirq.LineQubit(ghzqub1+2) , cirq.LineQubit(4*d2+8) , cirq.LineQubit(8*d2+4))
                    edited_cirq_rep.append(cirq.Moment([ccz1, ccz2, ccz3]))

                elif cy_moments % 3 == 1: #gate layer 2 diagonal gates
                    cs1 =  sevendiaggate( cirq.LineQubit(0)).controlled_by(cirq.LineQubit(ghzqub1))
                    csdag2 = sodddiaggate( cirq.LineQubit(2*d2+2)).controlled_by(cirq.LineQubit(ghzqub1+1))
                    cs3 = sevendiaggate( cirq.LineQubit(4*d2+4)).controlled_by(cirq.LineQubit(ghzqub1+2))
                    edited_cirq_rep.append(cirq.Moment([cs1, csdag2, cs3]))

                elif cy_moments % 3 == 2: #gate layer 3: 2 diags one off diag
                    csdag1 = sodddiaggate( cirq.LineQubit(6*d2+6)).controlled_by(cirq.LineQubit(ghzqub1))
                    cs2 = sevendiaggate( cirq.LineQubit(8*d2+8)).controlled_by(cirq.LineQubit(ghzqub1+1))
                    ccz3last = cirq.CCZ(cirq.LineQubit(ghzqub1+2) , cirq.LineQubit(8) , cirq.LineQubit(8*d2))
                    edited_cirq_rep.append(cirq.Moment([csdag1, cs2, ccz3last]))

                this_moment_is_CY = True

        elif op.gate == cirq.H and op.qubits[0].x == 2*(d2+1)**2 +1:
            if ghzh_counter % 2 == 1:
                edited_cirq_rep.append(cirq.Moment(cirq.ZPowGate(exponent=-0.25)(op.qubits[0])))
                new_moment.append(op) #("Adding T before meas")
            else:
                new_moment.append(op)
            ghzh_counter+=1

        else:
            new_moment.append(op)

    if this_moment_is_CY:
        cy_moments+=1
    edited_cirq_rep.append(cirq.Moment(new_moment))

In [10]:
if show_circuits:
    edited_cirq_rep

In [11]:
# filename = f"folded-cultivation-f{fault_distance}-d{final_distance}.json"
# cirq.to_json(cirq_rep, )

In [12]:
# cirq.optimize_for_target_gateset(edited_cirq_rep, gateset=cirq.CZTargetGateset())

In [13]:
def f(x):
    return x
serial = Counter()
parallel = Counter()
for moment in edited_cirq_rep[0:]:
    gates = [op.gate for op in moment.operations if op not in cirq.GateFamily(cirq.I)]
    if gates and gates[0] is not None:
        if gates[0] in cirq.GateFamily(cirq.MeasurementGate):
            gates = [cirq.MeasurementGate] * len(gates)
        if gates[0] in cirq.GateFamily(stimcirq.MeasureAndOrResetGate):
            gates = [cirq.MeasurementGate] * len(gates)
        if isinstance(gates[0], cirq.ControlledGate) and gates[0] not in cirq.GateFamily(cirq.CCZ):
            gates = [cirq.CNOT] * len(gates)
        serial += Counter(gates)
        parallel += Counter(gates[:1])
#     else:
#         print("This got skipped")
#         print(gates)
#         print(gates)
#     print("*"*50)
print(serial)
print(parallel)

Counter({cirq.CNOT: 2143, cirq.ResetChannel(): 993, cirq.H: 924, <class 'cirq.ops.measurement_gate.MeasurementGate'>: 895, cirq.CCZ: 21, cirq.S: 1})
Counter({cirq.CNOT: 77, cirq.H: 29, cirq.ResetChannel(): 16, <class 'cirq.ops.measurement_gate.MeasurementGate'>: 16, cirq.CCZ: 7, cirq.S: 1})
